## ライブラリの読み込み

In [ ]:
import random
from beamngpy import BeamNGpy, Scenario, Vehicle, set_up_simple_logging
from beamngpy.sensors import Camera
from IPython.display import display

## BeamNGの起動

In [ ]:
random.seed(1703)
set_up_simple_logging()

beamng = BeamNGpy('localhost', 25252)
bng = beamng.open(launch=False)

In [ ]:
vehicle_model = 'etk800'
map_name = 'c1'
spawn_pos = (3819.65, -5113.19, 852.5)
spawn_rot = (0.0, 0.0, -0.25881905, 0.96592583)

vehicle = Vehicle('ego_vehicle', model=vehicle_model, licence='ego_vehicle')

scenario = Scenario(map_name, 'LiDAR_demo', description='Spanning the map with a LiDAR sensor')

scenario.add_vehicle(vehicle, cling=True,
  pos=spawn_pos,
  rot_quat=spawn_rot
)

scenario.make(bng)
bng.settings.set_deterministic(60)
bng.load_scenario(scenario)
bng.ui.hide_hud()
bng.scenario.start()

## Camera

In [ ]:
camera = Camera(
    # センサーの一意な名前（複数カメラがある場合は区別用）
    name='transfuser_camera_front',
    
    # BeamNG との通信インスタンス（必須）
    bng=bng,
    
    # ============================================================================
    # 【取付先設定】
    # ============================================================================
    
    # カメラを取り付ける対象の車両
    # None の場合は固定カメラ（世界座標系で固定）
    # Vehicle オブジェクトの場合は車体に取り付け（自動追従）
    vehicle=vehicle,
    
    # ============================================================================
    # 【更新レート設定】
    # ============================================================================
    
    # シミュレーターがこのセンサーを更新するまでの待機時間（秒）
    # 0.1 = 10Hz（毎シミュレーションステップ実行）
    # -1 = 自動更新なし（poll/stream時にオンデマンド取得）
    # 値が小さいほど頻繁に更新（CPU負荷増加）
    requested_update_time=0.05,
    
    # センサー更新優先度 [0.0 ~ 1.0]
    # 0.0 = 最低優先度（他のセンサーがいっぱいなら後回し）
    # 1.0 = 最高優先度（優先的に更新）
    # 複数センサーがある場合は競合時の優先順位を制御
    update_priority=0.5,
    
    # ============================================================================
    # 【カメラの位置・姿勢】
    # ============================================================================
    
    # カメラの 3D 位置座標 (X, Y, Z) [メートル]
    # is_static=True の場合 → 世界座標系
    # is_static=False（vehicle指定） → 車両相対座標系
    # 例：(0, 0, 3) = 車の上方0.3m に配置
    pos=(0.0, -2.2, 0.5),
    
    # カメラの前方向ベクトル (X, Y, Z)
    # 正規化されている必要あり（ただし自動正規化される）
    # 例：(0, -1, 0) = Y軸負方向（前方）
    # is_dir_world_space=False の場合は車両座標系
    dir=(0.0, -1.0, 0.0),
    
    # カメラの上方向ベクトル (X, Y, Z)
    # dir と直交する必要がある
    # 例：(0, 0, 1) = Z軸正方向（上）
    up=(0.0, 0.0, 1.0),
    
    # ============================================================================
    # 【画像解像度・視野角】
    # ============================================================================
    
    # 出力画像の解像度 (幅, 高さ) [ピクセル]
    # 高いほど詳細だが処理時間・メモリ増加
    # 512x512 = 標準的（TransFuser用）
    # 1024x1024 = 高解像度（GPU負荷大）
    resolution=(704, 160),
    
    # 垂直視野角 [度]
    # 小さい → 望遠（狭い範囲を詳細）
    # 大きい → 広角（広い範囲を見える）
    # 70度 = 標準的な自動運転カメラ
    field_of_view_y=132,
    
    # ニアプレーン・ファープレーン [メートル]
    # (近い方, 遠い方)
    # ニア < この距離のオブジェクト = レンダリング対象外（クリップ）
    # ファー > この距離のオブジェクト = レンダリング対象外（クリップ）
    # (0.05, 100.0) = 5cm 〜 100m の範囲を描画
    near_far_planes=(0.1, 1000.0),
    
    # ============================================================================
    # 【レンダリング対象の選択】
    # ============================================================================
    
    # RGB カラー画像をレンダリングするか
    # True = 通常のRGB画像を生成
    # False = 出力しない（メモリ・処理時間削減）
    is_render_colours=True,
    
    # セマンティック注釈画像をレンダリングするか
    # True = オブジェクトクラス（車、歩行者、道路など）をマップ
    # False = 出力しない
    # TransFuser では必須（セマンティックセグメンテーション用）
    is_render_annotations=True,
    
    # インスタンス注釈画像をレンダリングするか
    # True = 個別オブジェクトのID をマップ（同じクラスでも違うID）
    # False = 出力しない
    # 複数オブジェクトの個別認識が必要な場合は有効
    is_render_instance=False,
    
    # 深度画像をレンダリングするか
    # True = 各ピクセルの深度値（距離）を生成
    # False = 出力しない
    # TransFuser では必須（深度推定用）
    is_render_depth=True,
    
    # ============================================================================
    # 【共有メモリ設定】高速データ転送用
    # ============================================================================
    
    # 共有メモリを使用するか
    # True = CPU-GPU 間のデータ転送を高速化
    #        （stream() 使用時に重要）
    # False = 通常の通信（遅い）
    # stream() を使う場合は True 推奨
    is_using_shared_memory=True,
    
    # ============================================================================
    # 【深度値の表現方法】
    # ============================================================================
    
    # 深度値を反転するか
    # False = 黒(近) → 白(遠) の自然な勾配
    # True = 白(近) → 黒(遠) に反転
    # 一般的には False
    is_depth_inverted=False,
    
    # 深度値を後処理するか
    # True = 非線形スケーリングで中間値を強調
    #        （小さな距離差をより見やすくする）
    #        ただし計算負荷が大きい
    # False = 線形スケーリング（高速）
    # TransFuser では False 推奨（高速性重要）
    postprocess_depth=False,
    
    # ============================================================================
    # 【ストリーミング・共有メモリ設定】
    # ============================================================================
    
    # ストリーミングモードを有効にするか
    # True = stream() メソッドで高速取得可能
    #        シミュレーターが自動でデータを共有メモリに書き込み
    #        poll() は使えない（stream() のみ）
    # False = poll()/poll_raw() で都度リクエスト送信
    # 大量フレーム取得には True 必須！
    is_streaming=True,
    
    # ============================================================================
    # 【カメラの固定/移動】
    # ============================================================================
    
    # カメラを固定にするか
    # True = 世界座標系で固定位置
    #        pos, dir, up は世界座標
    # False = 車両に取り付け（vehicle パラメータ利用）
    #        pos, dir, up は車両相対座標
    # TransFuser では False（車両に取り付け）
    is_static=False,
    
    # ============================================================================
    # 【座標系設定】
    # ============================================================================
    
    # カメラの方向ベクトル (dir) が世界座標系か車両座標系か
    # True = dir/up は世界座標系（固定方向）
    # False = dir/up は車両座標系（車と共に回転）
    # is_static=False の場合は False 推奨
    is_dir_world_space=False,
    
    # ============================================================================
    # 【ビジュアライゼーション】デバッグ用
    # ============================================================================
    
    # シミュレーター画面にカメラを表示するか
    # True = シミュレーター画面でカメラの視野を表示（デバッグ用）
    # False = 非表示（処理には影響なし）
    is_visualised=False,
    
    # ============================================================================
    # 【車両への吸着・埋め込み設定】is_static=False 時のみ有効
    # ============================================================================
    
    # 最も近い車両ポリゴンに吸着させるか
    # True = センサーを最も近いポリゴンに自動吸着
    # False = 吸着なし（精密配置が可能）
    # 通常は True（自動で適切に配置）
    is_snapping_desired=True,
    
    # センサーを三角形内に強制埋め込みするか
    # True = 最も近い三角形の内側に強制
    # False = 表面に配置
    # 通常は False（表面配置）
    is_force_inside_triangle=False,
)

In [ ]:
data = camera.stream()

colour_img = data['colour']
annot_img = data['annotation']
depth_img = data['depth']

print("Colour:")
display(colour_img)

print("Annotation:")
display(annot_img)

print("Depth:")
display(depth_img)

## LiDAR